# Telecom Churn — Data Audit

## Purpose
Inspect the development dataset, document its structure and quality,
and identify potential issues before exploratory analysis and modelling.

## Data source
Telecom Churn Dataset (BigML split), publicly available on Kaggle:
https://www.kaggle.com/datasets/mnassrib/telecom-churn-datasets

The dataset is supplied as two files:
- `churn-bigml-80.csv` — 2,666 customer accounts
- `churn-bigml-20.csv` — 667 customer accounts

Each record contains 19 input variables (account, service plan, usage,
charge and customer-service information) and a binary `Churn` target.
Observation dates and the prediction horizon are not documented.

## Validation boundary
- The 80% file is the **development set**: used for auditing, EDA,
  feature decisions, model tuning and threshold selection.
- The 20% file is the **final test set**: in this notebook it is only
  checked for schema consistency and overlap with the development set.
  It is not explored or used for any modelling decision until final evaluation.

## 1. Setup
Connect to Google Drive and define the project root, so that all paths are relative to the project folder.

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/bda-churn-capstone')
except ImportError:
    PROJECT = Path.cwd().parent  # notebooks/ -> repository root

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load development data
Only the 80% development file is loaded. An unchanged copy is kept as `df_raw`; all work is done on `df`.

In [2]:
import pandas as pd

RAW = PROJECT / 'data' / 'raw'
DEV_PATH = RAW / 'churn-bigml-80.csv'
TEST_PATH = RAW / 'churn-bigml-20.csv'

for path in (DEV_PATH, TEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Missing data file: {path}")

pd.set_option('display.max_columns', None)

# Load development data; preserve an unchanged copy.
df_raw = pd.read_csv(DEV_PATH)
df = df_raw.copy()

print(f"Development file: {DEV_PATH.name}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

display(df.head())

Development file: churn-bigml-80.csv
Rows: 2,666
Columns: 20


,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


## 3. Variable dictionary
Documents the meaning, group, role and data type of each variable, with notes on issues to check later.

In [3]:
# Variable dictionary: meaning, group, role and audit notes for each column
descriptions = {
    'State': ('US state (two-letter code)', 'Account', 'Feature',
              '51 categories; high cardinality for one-hot encoding'),
    'Account length': ('Length of the customer account (unit not documented)', 'Account', 'Feature', ''),
    'Area code': ('Telephone area code', 'Account', 'Feature',
                  'Stored as number but is categorical'),
    'International plan': ('Has international calling plan (Yes/No)', 'Plan', 'Feature', ''),
    'Voice mail plan': ('Has voice mail plan (Yes/No)', 'Plan', 'Feature', ''),
    'Number vmail messages': ('Number of voice mail messages', 'Usage', 'Feature', ''),
    'Total day minutes': ('Total daytime call minutes', 'Usage', 'Feature', ''),
    'Total day calls': ('Total daytime calls', 'Usage', 'Feature', ''),
    'Total day charge': ('Total daytime charge', 'Charge', 'Feature',
                         'Check redundancy with day minutes'),
    'Total eve minutes': ('Total evening call minutes', 'Usage', 'Feature', ''),
    'Total eve calls': ('Total evening calls', 'Usage', 'Feature', ''),
    'Total eve charge': ('Total evening charge', 'Charge', 'Feature',
                         'Check redundancy with eve minutes'),
    'Total night minutes': ('Total night call minutes', 'Usage', 'Feature', ''),
    'Total night calls': ('Total night calls', 'Usage', 'Feature', ''),
    'Total night charge': ('Total night charge', 'Charge', 'Feature',
                           'Check redundancy with night minutes'),
    'Total intl minutes': ('Total international call minutes', 'Usage', 'Feature', ''),
    'Total intl calls': ('Total international calls', 'Usage', 'Feature', ''),
    'Total intl charge': ('Total international charge', 'Charge', 'Feature',
                          'Check redundancy with intl minutes'),
    'Customer service calls': ('Number of calls to customer service', 'Service', 'Feature',
                               'Possible leakage: timing relative to churn unknown'),
    'Churn': ('Customer left the service (True/False)', 'Target', 'Target', 'Binary target'),
}

variable_table = pd.DataFrame({
    'Variable': df.columns,
    'Description': [descriptions[c][0] for c in df.columns],
    'Group': [descriptions[c][1] for c in df.columns],
    'Role': [descriptions[c][2] for c in df.columns],
    'Data type': df.dtypes.astype(str).values,
    'Unique values': df.nunique().values,
    'Missing': df.isna().sum().values,
    'Example': df.iloc[0].astype(str).values,
    'Audit note': [descriptions[c][3] for c in df.columns],
})

display(variable_table)

# Save for use in the report
variable_table.to_csv(PROJECT / 'reports' / 'tables' / 'variable_dictionary.csv', index=False)
print("Saved: reports/tables/variable_dictionary.csv")

,Variable,Description,Group,Role,Data type,Unique values,Missing,Example,Audit note
0,State,US state (two-letter code),Account,Feature,object,51,0,KS,51 categories; high cardinality for one-hot en...
1,Account length,Length of the customer account (unit not docum...,Account,Feature,int64,205,0,128,
2,Area code,Telephone area code,Account,Feature,int64,3,0,415,Stored as number but is categorical
3,International plan,Has international calling plan (Yes/No),Plan,Feature,object,2,0,No,
4,Voice mail plan,Has voice mail plan (Yes/No),Plan,Feature,object,2,0,Yes,
5,Number vmail messages,Number of voice mail messages,Usage,Feature,int64,42,0,25,
6,Total day minutes,Total daytime call minutes,Usage,Feature,float64,1489,0,265.1,
7,Total day calls,Total daytime calls,Usage,Feature,int64,115,0,110,
8,Total day charge,Total daytime charge,Charge,Feature,float64,1489,0,45.07,Check redundancy with day minutes
9,Total eve minutes,Total evening call minutes,Usage,Feature,float64,1442,0,197.4,


Saved: reports/tables/variable_dictionary.csv


**Findings**
- The dataset has 19 input variables and one binary target (`Churn`).
- `Area code` is stored as a number but has only three values and is categorical.
- `State` has 51 categories, which makes one-hot encoding high-dimensional.
- Each charge variable may duplicate its corresponding minutes variable; this is checked in the next steps.
- `Customer service calls` may carry leakage risk, because its timing relative to churn is not documented.

## 4. Data quality audit
Checks data types, missing values, blank strings, unique values and duplicate rows in the development data.

In [4]:
# Basic quality audit of the development data.
audit = pd.DataFrame({
    'Data type': df.dtypes.astype(str),
    'Missing values': df.isna().sum(),
    'Unique values': df.nunique(dropna=True),
    'Blank strings': [
        df[col].astype('string').str.strip().eq('').sum()
        for col in df.columns
    ]
})

audit.index.name = 'Column'
display(audit)

print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"\nTarget values: {df['Churn'].unique()}")

,Data type,Missing values,Unique values,Blank strings
Column,,,,
State,object,0,51,0
Account length,int64,0,205,0
Area code,int64,0,3,0
International plan,object,0,2,0
Voice mail plan,object,0,2,0
Number vmail messages,int64,0,42,0
Total day minutes,float64,0,1489,0
Total day calls,int64,0,115,0
Total day charge,float64,0,1489,0


Exact duplicate rows: 0

Target values: [False  True]


**Findings**
- No missing values or blank strings were found.
- No exact duplicate rows were found.
- The target `Churn` takes only two values (True/False), so no cleaning of the target is needed.

## 5. Schema and overlap check with the test file
The 20% test file is loaded only to confirm that it has the same structure as the development file and that no identical record appears in both files. The dataset has no customer identifier, so overlap can only be checked at the level of exact record matches. No statistics or patterns of the test data are explored.

In [5]:
# Load the test file only for schema and overlap checks.
test_audit = pd.read_csv(TEST_PATH)
print(f"Test file: {TEST_PATH.name} | Rows: {test_audit.shape[0]:,} | Columns: {test_audit.shape[1]}")

# 1. Compare column names, order and inferred data types.
same_columns = df_raw.columns.equals(test_audit.columns)

schema_check = pd.DataFrame({
    'Development dtype': df_raw.dtypes.astype(str),
    'Test dtype': test_audit.dtypes.astype(str)
})
schema_check['Match'] = (
    schema_check['Development dtype'] == schema_check['Test dtype']
)

print(f"Same column names and order: {same_columns}")
display(schema_check)

if not same_columns:
    raise ValueError("Column names or order differ. Review before continuing.")

# 2. Count shared distinct records, including the target.
full_overlap = (
    df_raw.drop_duplicates()
    .merge(test_audit.drop_duplicates(), how='inner')
    .shape[0]
)

# 3. Check identical predictor profiles, regardless of target.
features = df_raw.columns.drop('Churn').tolist()

predictor_overlap = (
    df_raw[features].drop_duplicates()
    .merge(
        test_audit[features].drop_duplicates(),
        on=features,
        how='inner'
    )
    .shape[0]
)

print(f"\nShared distinct full records: {full_overlap}")
print(f"Shared distinct predictor profiles: {predictor_overlap}")

# Retain only the audit results, not the test dataframe.
del test_audit

if full_overlap or predictor_overlap:
    print("\nOverlap detected — review before modelling.")
else:
    print("\nNo exact overlap detected.")

Test file: churn-bigml-20.csv | Rows: 667 | Columns: 20
Same column names and order: True


,Development dtype,Test dtype,Match
State,object,object,True
Account length,int64,int64,True
Area code,int64,int64,True
International plan,object,object,True
Voice mail plan,object,object,True
Number vmail messages,int64,int64,True
Total day minutes,float64,float64,True
Total day calls,int64,int64,True
Total day charge,float64,float64,True
Total eve minutes,float64,float64,True



Shared distinct full records: 0
Shared distinct predictor profiles: 0

No exact overlap detected.


**Findings**
- The test file has the same column names, order and data types as the development file.
- No shared full records were found between the two files.
- No shared predictor profiles were found either, so no test record has an identical feature set in the development data, regardless of its churn label.
- Because there is no customer identifier, it cannot be fully ruled out that the same customer appears in both files with different values; this is noted as a limitation.
- The test file is now set aside until final evaluation.

## 6. Value validity checks
Check the development data for negative or non-finite numerical
values, non-integer counts, and unexpected categorical values.
Flagged values will be reviewed before any cleaning decisions.

In [6]:
import numpy as np

# Check numerical values in development data only.
numeric_cols = df_raw.select_dtypes(include='number').columns

validity = pd.DataFrame({
    'Minimum': df_raw[numeric_cols].min(),
    'Maximum': df_raw[numeric_cols].max(),
    'Negative values': df_raw[numeric_cols].lt(0).sum(),
    'Infinite values': np.isinf(df_raw[numeric_cols]).sum()
})

display(validity)

# Counts and account length should contain whole numbers.
count_cols = [
    'Account length',
    'Number vmail messages',
    'Total day calls',
    'Total eve calls',
    'Total night calls',
    'Total intl calls',
    'Customer service calls'
]

print("Non-integer values in count columns:")
display(pd.Series({
    col: (
        df_raw[col].notna() & df_raw[col].mod(1).ne(0)
    ).sum()
    for col in count_cols
}, name='Non-integer count').to_frame())

# Inspect category labels without changing them.
print("\nObserved categorical values:")
for col in ['State', 'Area code', 'International plan',
            'Voice mail plan', 'Churn']:
    print(f"{col}: {df_raw[col].dropna().unique().tolist()}")

,Minimum,Maximum,Negative values,Infinite values
Account length,1.00,243.00,0,0
Area code,408.00,510.00,0,0
Number vmail messages,0.00,50.00,0,0
Total day minutes,0.00,350.80,0,0
Total day calls,0.00,160.00,0,0
Total day charge,0.00,59.64,0,0
Total eve minutes,0.00,363.70,0,0
Total eve calls,0.00,170.00,0,0
Total eve charge,0.00,30.91,0,0
Total night minutes,43.70,395.00,0,0


Non-integer values in count columns:


,Non-integer count
Account length,0
Number vmail messages,0
Total day calls,0
Total eve calls,0
Total night calls,0
Total intl calls,0
Customer service calls,0



Observed categorical values:
State: ['KS', 'OH', 'NJ', 'OK', 'AL', 'MA', 'MO', 'WV', 'RI', 'IA', 'MT', 'ID', 'VT', 'VA', 'TX', 'FL', 'CO', 'AZ', 'NE', 'WY', 'IL', 'NH', 'LA', 'GA', 'AK', 'MD', 'AR', 'WI', 'OR', 'DE', 'IN', 'UT', 'CA', 'SD', 'NC', 'WA', 'MN', 'NM', 'NV', 'DC', 'NY', 'KY', 'ME', 'MS', 'MI', 'SC', 'TN', 'PA', 'HI', 'ND', 'CT']
Area code: [415, 408, 510]
International plan: ['No', 'Yes']
Voice mail plan: ['Yes', 'No']
Churn: [False, True]


**Findings**
- No negative or infinite values were found in the numerical columns.
- All checked count variables contained whole-number values.
- Both plan variables contained Yes/No values, and Churn contained Boolean values.
- Area code contained three distinct codes and will be treated as a categorical feature.
- All three area codes (408, 415, 510) are California codes, while State covers 51 states. Whether this indicates an inconsistency has not been examined in the data; the handling of location variables is reviewed during EDA.
- These checks did not identify a reason to remove records. Zero values and extreme observations require contextual assessment.

## 7. Charge–minutes redundancy check
Each charge variable is compared with its corresponding minutes variable using correlation and a simple linear fit. If charge is a fixed rate multiplied by minutes, it adds no independent information. Only development data is used, and the target is not involved.

In [7]:
# Assess charge–minutes redundancy using development data only.
pairs = [
    ('Total day minutes', 'Total day charge'),
    ('Total eve minutes', 'Total eve charge'),
    ('Total night minutes', 'Total night charge'),
    ('Total intl minutes', 'Total intl charge')
]

results = []

for minutes_col, charge_col in pairs:
    pair = df_raw[[minutes_col, charge_col]].dropna()
    x = pair[minutes_col].to_numpy()
    y = pair[charge_col].to_numpy()

    slope, intercept = np.polyfit(x, y, 1)
    residuals = y - (slope * x + intercept)

    results.append({
        'Minutes variable': minutes_col,
        'Charge variable': charge_col,
        'Correlation': pair[minutes_col].corr(pair[charge_col]),
        'Estimated rate': slope,
        'Intercept': intercept,
        'Maximum absolute residual': np.abs(residuals).max()
    })

redundancy_check = pd.DataFrame(results)
display(redundancy_check.round(6))

redundancy_check.to_csv(PROJECT / 'reports' / 'tables' / 'charge_minutes_redundancy.csv', index=False)
print("Saved: reports/tables/charge_minutes_redundancy.csv")

,Minutes variable,Charge variable,Correlation,Estimated rate,Intercept,Maximum absolute residual
0,Total day minutes,Total day charge,1.000000,0.170000,0.000619,0.004594
1,Total eve minutes,Total eve charge,1.000000,0.085000,0.000310,0.004793
2,Total night minutes,Total night charge,0.999999,0.045000,0.000074,0.005099
3,Total intl minutes,Total intl charge,0.999993,0.269983,0.000669,0.004631


Saved: reports/tables/charge_minutes_redundancy.csv


**Findings and modelling decision**

All four charge variables were almost perfectly linearly related to their corresponding call-minute variables (correlation ≈ 1.000, intercept ≈ 0). Estimated rates were approximately 0.170, 0.085, 0.045 and 0.270 for daytime, evening, night-time and international calls, respectively.

The maximum residuals (about 0.005) are consistent with rounding charges to the nearest cent. The four charge variables will therefore be excluded from model inputs to reduce redundancy, reduce multicollinearity in Logistic Regression and simplify interpretation, while the corresponding minute variables will be retained.

This decision is based only on the development data. The raw files remain unchanged; the exclusion is applied in the modelling pipeline. Redundancy alone is not evidence of target leakage.

## 8. Leakage assessment and sensitivity plan

The dataset does not document observation dates, a prediction cut-off or the timing of individual features relative to churn. It is therefore not possible to confirm that all predictors were available before the outcome occurred.

Customer service calls requires particular caution: the recorded count could include contacts associated with cancellation. This is a potential risk, not confirmed leakage.

The table below summarises the assessment for each variable group.

In [8]:
# Conceptual assessment based on feature meanings.
# No target associations or test-data patterns are used.
leakage_table = pd.DataFrame([
    [
        'Account',
        'State, Account length, Area code',
        'Not verified',
        'Account attributes may be available before churn, but the '
        'recording date and prediction cut-off are undocumented.',
        'Retain; acknowledge timing uncertainty.'
    ],
    [
        'Plan',
        'International plan, Voice mail plan',
        'Not verified',
        'The data do not establish whether plan status was recorded '
        'before or after the intended prediction cut-off.',
        'Retain; acknowledge timing uncertainty.'
    ],
    [
        'Usage',
        'Day/eve/night/intl minutes and calls, vmail messages',
        'Not verified',
        'The aggregation window is undocumented. Usage is valid only '
        'if measured before the prediction cut-off.',
        'Retain; acknowledge the unknown observation window.'
    ],
    [
        'Charge',
        'Day/eve/night/intl charge',
        'Not verified',
        'Charges share the timing uncertainty of usage. Redundancy '
        'with minutes is not itself evidence of leakage.',
        'Exclude from model inputs because of redundancy.'
    ],
    [
        'Service',
        'Customer service calls',
        'Not verified; specific concern',
        'The count could include cancellation-related contacts; '
        'their timing relative to churn is unknown.',
        'Retain in the main analysis; repeat modelling without '
        'this feature as a predefined sensitivity analysis.'
    ]
], columns=[
    'Group',
    'Variables',
    'Availability before prediction cut-off',
    'Timing concern',
    'Planned treatment'
])

# Show full text in table cells
pd.set_option('display.max_colwidth', None)
display(leakage_table)

tables_dir = PROJECT / 'reports' / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)

leakage_table.to_csv(
    tables_dir / 'leakage_assessment.csv',
    index=False
)

print("Saved: reports/tables/leakage_assessment.csv")

,Group,Variables,Availability before prediction cut-off,Timing concern,Planned treatment
0,Account,"State, Account length, Area code",Not verified,"Account attributes may be available before churn, but the recording date and prediction cut-off are undocumented.",Retain; acknowledge timing uncertainty.
1,Plan,"International plan, Voice mail plan",Not verified,The data do not establish whether plan status was recorded before or after the intended prediction cut-off.,Retain; acknowledge timing uncertainty.
2,Usage,"Day/eve/night/intl minutes and calls, vmail messages",Not verified,The aggregation window is undocumented. Usage is valid only if measured before the prediction cut-off.,Retain; acknowledge the unknown observation window.
3,Charge,Day/eve/night/intl charge,Not verified,Charges share the timing uncertainty of usage. Redundancy with minutes is not itself evidence of leakage.,Exclude from model inputs because of redundancy.
4,Service,Customer service calls,Not verified; specific concern,The count could include cancellation-related contacts; their timing relative to churn is unknown.,Retain in the main analysis; repeat modelling without this feature as a predefined sensitivity analysis.


Saved: reports/tables/leakage_assessment.csv


**Decision and sensitivity plan**

The main analysis will retain Customer service calls. A predefined sensitivity analysis will repeat model development and evaluation without this feature, using the same validation design and tuning budget. Each version will be tuned independently using development data only.

Removing this feature does not establish that the remaining predictors are temporally valid. Results will be interpreted as discrimination of recorded churn labels, with prospective performance requiring further validation.

Churn will be excluded from all predictor sets. Learned preprocessing will be fitted within training folds, and the test set will not inform feature selection, tuning or threshold selection.

## 9. Audit summary and recorded decisions
All decisions made in this notebook are collected in one place and saved as a JSON file, so that later notebooks (EDA and modelling) use exactly the same feature definitions, sensitivity plan and documented limitations.

In [9]:
import json

target = 'Churn'
charge_cols = [
    'Total day charge', 'Total eve charge',
    'Total night charge', 'Total intl charge'
]
categorical_cols = [
    'State', 'Area code',
    'International plan', 'Voice mail plan'
]
numeric_cols = [
    c for c in df_raw.columns
    if c not in charge_cols + categorical_cols + [target]
]

audit_decisions = {
    'development_file': DEV_PATH.name,
    'test_file': TEST_PATH.name,
    'target': target,
    'rows_removed': 0,
    'excluded_features': charge_cols,
    'exclusion_reason': (
        'Charge variables are near-exact linear functions of '
        'minutes (Section 7); excluded for redundancy, not leakage.'
    ),
    'categorical_features': categorical_cols,
    'numeric_features': numeric_cols,
    'sensitivity_analysis': {
        'feature': 'Customer service calls',
        'main_analysis': 'Feature retained.',
        'sensitivity_version': (
            'Model development and evaluation repeated '
            'without this feature.'
        ),
        'conditions': (
            'Same validation design and tuning budget; each version '
            'tuned independently on development data only.'
        ),
    },
    'preprocessing_policy': (
        'Fit learned preprocessing within training folds only.'
    ),
    'test_policy': (
        'Before final evaluation, use the test file only for '
        'schema and exact-overlap checks. Do not use it for EDA, '
        'feature selection, tuning or threshold selection.'
    ),
    'limitations': [
        'Observation dates, prediction cut-off and feature timing '
        'are not documented.',
        'Availability of predictors before churn is not verified.',
        'Customer service calls may include cancellation-related '
        'contacts.',
        'No customer identifier; overlap was checked only for exact '
        'records and identical predictor profiles.',
        'Results describe discrimination of recorded churn labels, '
        'not prospective performance or causal effects.',
    ],
    'open_questions_for_eda': [
        'Handling of State (51 categories; high cardinality).',
        'Relationship between Voice mail plan and '
        'Number vmail messages.',
    ],
}

processed_dir = PROJECT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
decisions_path = processed_dir / 'audit_decisions.json'

with decisions_path.open('w', encoding='utf-8') as f:
    json.dump(audit_decisions, f, indent=2, ensure_ascii=False)

summary = pd.DataFrame({
    'Item': [
        'Development rows', 'Rows removed',
        'Model features (main)', 'Categorical features',
        'Numeric features', 'Excluded features',
        'Sensitivity feature', 'Recorded limitations'
    ],
    'Value': [
        len(df_raw), 0,
        len(categorical_cols) + len(numeric_cols),
        len(categorical_cols), len(numeric_cols),
        len(charge_cols), 'Customer service calls',
        len(audit_decisions['limitations'])
    ],
})
display(summary)
print(f"Saved: {decisions_path.relative_to(PROJECT)}")

,Item,Value
0,Development rows,2666
1,Rows removed,0
2,Model features (main),15
3,Categorical features,4
4,Numeric features,11
5,Excluded features,4
6,Sensitivity feature,Customer service calls
7,Recorded limitations,5


Saved: data/processed/audit_decisions.json


## 10. Next steps

The checks performed identified no quality or validity issues requiring record removal, and no exact overlap was found between the development and test files. All development records were retained. The documented timing and customer-identification limitations remain.

The following decisions are carried forward: the four charge variables are excluded as redundant, Area code is treated as categorical, and Customer service calls is retained in the main analysis with a predefined sensitivity analysis.

The next notebook (`02_eda.ipynb`) will explore the development data only, using the decisions saved in `data/processed/audit_decisions.json`. It will examine the churn distribution and class balance, the relationships between individual features and churn, State category frequencies, and the relationship between Voice mail plan and Number vmail messages. The test data will remain reserved for final evaluation.